# Session 8: Multi-Class and Hierarchical Text Classification

## Overview

Sessions 6 and 7 had two classes. Real text often has many: support tickets go to one of a dozen teams, news
articles to one of many sections, posts to one of many forums. Here you sort posts from 20 Usenet newsgroups into
their **20 topics**. Some topics are easy to tell apart (hockey against cryptography); others are near neighbours
(two Windows groups, three religion groups). The topics also form a **hierarchy**: `rec.sport.hockey` belongs to
`rec` (recreation), `comp.graphics` to `comp` (computers), and so on.

By the end, you will be able to:

- Explain how binary classifiers are extended to many classes (one-vs-rest).
- Train and evaluate a multi-class text classifier: accuracy, macro-averaged F1, per-class reports.
- Read a large confusion matrix and find the classes the model confuses.
- Compare a flat classifier with a top-down hierarchical one, and explain why errors propagate down a hierarchy.
- Recognise metadata leakage in text (email headers that name the class).
- Describe multi-label classification and how it differs from multi-class.

**Prerequisites:** Session 7 (TF-IDF, pipelines, `LinearSVC`).

**Dataset:** [20 Newsgroups](https://archive.ics.uci.edu/dataset/113/twenty+newsgroups) (also in the UCI
repository), loaded with scikit-learn's `fetch_20newsgroups`. 18,846 posts from 20 Usenet discussion groups in
1993, in the standard "by date" split: the training posts were written **before** the test posts, so the test
imitates classifying future posts. The first run downloads about 14 MB.

## Where this fits

| Part | Sessions | Task | Algorithms |
| --- | --- | --- | --- |
| 1. Regression | 1, 2, 3, 4 | Predict a number | Linear regression by gradient descent and in scikit-learn, tree ensembles, explaining black-box models |
| 2. Classification | 5, 6, 7, 8, 9 | Predict a category | Logistic regression, naive Bayes, linear text classifiers, comparing classifiers **← you are here** |
| 3. Clustering | 10, 11 | Find groups without labels | k-means, hierarchical clustering |
| 4. Topic modelling | 12 | Find themes in text | LDA, NMF |

**This session's question:** How well can a linear model sort posts into 20 topics, and which topics does it
confuse?

## Step 1: Set up the environment

This module has one pinned `requirements.txt` (in the module folder) for all twelve sessions. If you have
not done it yet, create a virtual environment and install it once, from the module folder:

```bash
python3.11 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\Activate.ps1
pip install -r requirements.txt
jupyter notebook
```

This session uses:

| Package | Why we need it |
| --- | --- |
| scikit-learn | Dataset download, vectoriser, classifiers, metrics |
| pandas | Tables |
| numpy | Arrays |
| matplotlib | Charts |

Run the cell below. It imports what this notebook needs and prints the versions. The numbers quoted in
this notebook come from these exact versions; a different version can shift the last decimal place.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC

from importlib.metadata import version

for package in ["scikit-learn", "pandas", "numpy", "matplotlib"]:
    print(f"{package:13s} {version(package)}")

## Step 2: Load the posts, without the giveaways

Each raw post starts with email-style **headers**, and many end with a signature (**footer**) and contain quoted
lines from earlier posts (**quotes**). Look at one raw post first.

In [ ]:
raw = fetch_20newsgroups(subset="train")
print(raw.data[0][:420])

**Read the output:** the header includes the poster's email address, organisation and `Subject` line. In some
groups the same few people post constantly, so a model could learn to recognise **who** wrote a post rather than
**what** it is about. That would fail on posts by new people, and on text that has no headers at all (a support
ticket, a review). Step 10 measures how much this leak inflates the score.

We load the posts with headers, footers and quotes removed, so the model has to work from the content.

In [ ]:
STRIP = ("headers", "footers", "quotes")
train = fetch_20newsgroups(subset="train", remove=STRIP)
test = fetch_20newsgroups(subset="test", remove=STRIP)
names = np.array(train.target_names)
print(f"train: {len(train.data):,} posts, test: {len(test.data):,} posts, {len(names)} classes")

counts = pd.Series(names[train.target]).value_counts()
print(f"posts per class in training: {counts.min()} to {counts.max()}")
print(f"empty after stripping: {sum(not d.strip() for d in train.data)}")

**Read the output:** 11,314 training posts and 7,532 test posts. The classes are roughly balanced (377 to 600
training posts each), which keeps accuracy meaningful. Stripping leaves 300 training posts empty: their whole
content was a header, a quote or a signature. We keep them, since the same will happen in the test set; the model
simply cannot classify them well.

## Step 3: From two classes to twenty

Logistic regression and linear SVMs are binary by nature. scikit-learn extends them automatically:

- **One-vs-rest (OvR):** train 20 binary classifiers, "is it `rec.autos` or not?", "is it `sci.space` or not?",
  and so on. A new post goes to the class whose classifier gives the highest score. `LinearSVC` does this.
- **Multinomial models** handle all classes at once. Naive Bayes is naturally multi-class: it just compares 20
  class scores instead of 2. `ComplementNB` is a variant designed for text, which usually beats `MultinomialNB`.

Two summary metrics:

- **accuracy:** the share of posts given the right topic.
- **macro-averaged F1:** compute F1 for each class separately, then average with equal weight. A model that does
  well on big classes and badly on small ones gets a low macro F1, even when accuracy is high.

In [ ]:
def vectorizer():
    return TfidfVectorizer(sublinear_tf=True, min_df=2, stop_words="english")


models = {"ComplementNB": make_pipeline(vectorizer(), ComplementNB(alpha=0.3)),
          "LinearSVC (one-vs-rest)": make_pipeline(vectorizer(), LinearSVC())}
predictions = {}
for name, model in models.items():
    model.fit(train.data, train.target)
    predictions[name] = model.predict(test.data)
    print(f"{name:24s} accuracy = {accuracy_score(test.target, predictions[name]):.3f}   "
          f"macro F1 = {f1_score(test.target, predictions[name], average='macro'):.3f}")

svm = models["LinearSVC (one-vs-rest)"]
print(f"\n{len(svm[-1].coef_)} sets of weights (one per class), each over "
      f"{len(svm[0].vocabulary_):,} words")

**Read the output:** both models get about 70% of posts right (ComplementNB 0.711, LinearSVC 0.692; macro F1
0.693 and 0.681). Steps 4-7 use ComplementNB, the better of the two; Steps 8 and 10 use the SVM. With 20
classes, guessing would get 5%. The SVM has 20 sets of weights, one per one-vs-rest classifier.

`min_df=2` drops words seen in only one training post (mostly typos and names), and `stop_words="english"` drops
very common words such as "the" and "and".

70% may sound modest after the 99% spam filter, but this is a much harder problem: 20 classes, several of them
near-duplicates, from short, informal posts with the giveaway metadata removed.

## Step 4: Which classes are hard? The per-class report

In [ ]:
best = predictions["ComplementNB"]
report = pd.DataFrame(classification_report(test.target, best, target_names=names, output_dict=True)).T
print(report.iloc[:20].sort_values("f1-score").round(3).to_string())

**Read the output:** the best classes have distinctive vocabularies: `rec.sport.hockey` (F1 0.90),
`rec.sport.baseball` (0.88) and `talk.politics.mideast` (0.80). The worst are `talk.religion.misc` (0.23: it finds
only 15% of these posts), `alt.atheism` (0.37) and `talk.politics.misc` (0.52), all of which share their
vocabulary with a neighbouring group.

Look at `soc.religion.christian`: recall 0.89 but precision only 0.57. It catches its own posts, and also absorbs
many posts from the other two religion groups. The confusion matrix shows exactly where each class's posts go.

## Step 5: The confusion matrix

With 20 classes the confusion matrix has 400 cells. Normalising each row (dividing by the number of posts in the
true class) makes rows comparable: the diagonal is each class's recall.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 10))
ConfusionMatrixDisplay.from_predictions(names[test.target], names[best], normalize="true", values_format=".1f",
                                        xticks_rotation=90, ax=ax, colorbar=False, cmap="Blues",
                                        text_kw={"fontsize": 6})
ax.set_title("Confusion matrix, rows normalised (ComplementNB)")
plt.show()

table = pd.crosstab(names[test.target], names[best])
off_diagonal = table.where(~np.eye(len(names), dtype=bool)).stack().sort_values(ascending=False)
print("Most common confusions (true -> predicted):")
print(off_diagonal.head(8).astype(int).to_string())

**Read the output:** the diagonal is dark, but there are clear off-diagonal blocks: the **computer** groups
confuse each other (Windows, IBM PC hardware, graphics, X Window), and so do the **religion** groups
(`talk.religion.misc` and `alt.atheism` are often predicted as `soc.religion.christian`) and the **politics**
groups. The three biggest confusions are `talk.religion.misc` → `soc.religion.christian` (102 posts),
`talk.politics.misc` → `talk.politics.guns` (89) and `alt.atheism` → `soc.religion.christian` (87). One common
confusion crosses topics: `misc.forsale` → `comp.sys.ibm.pc.hardware` (32), because many for-sale posts sell PC
parts.

Most confusions are not random; they follow the topic hierarchy. Many posts genuinely could belong to either
group: the boundaries between Usenet groups were set by people, not by content.

## Step 6: The hierarchy

Group the 20 newsgroups into 6 broader topics. If a model confuses two Windows groups, it still knows the post is
about computers.

In [ ]:
def top_level(group):
    if group.startswith("comp."):
        return "computers"
    if group.startswith("rec."):
        return "recreation"
    if group.startswith("sci."):
        return "science"
    if group.startswith("talk.politics"):
        return "politics"
    if group in ("alt.atheism", "soc.religion.christian", "talk.religion.misc"):
        return "religion"
    return "for sale"                                   # misc.forsale


to_top = np.vectorize(top_level)
top_train, top_test = to_top(names[train.target]), to_top(names[test.target])
print(pd.Series(top_train).value_counts().to_string())

flat_top = to_top(names[best])
print(f"\nflat model, judged on the 6 top-level topics: accuracy = {accuracy_score(top_test, flat_top):.3f}")
errors = names[best] != names[test.target]
same_top = flat_top[errors] == top_test[errors]
print(f"of its {errors.sum():,} errors, {same_top.mean():.0%} stay within the right top-level topic")

**Read the output:** judged only on the 6 broad topics, the same predictions are 83.2% correct. Of the 2,173 leaf
errors, 42% are near misses that stay inside the right broad topic, such as one computer group mistaken for another.
For many uses, routing a post to the right broad area is already valuable.

## Step 7: Flat or top-down?

Two ways to use a hierarchy:

- **Flat:** one classifier over all 20 leaf classes, as so far. The top-level topic is read off the prediction.
- **Top-down (local classifier per parent node):** a first classifier picks the broad topic; then a separate
  classifier, trained only on that topic's posts, picks the leaf. Each second-level classifier only has to tell
  apart a few similar groups.

In [ ]:
documents_train = np.array(train.data, dtype=object)
documents_test = np.array(test.data, dtype=object)

top_model = make_pipeline(vectorizer(), ComplementNB(alpha=0.3)).fit(documents_train, top_train)
top_predicted = top_model.predict(documents_test)
print(f"stage 1, broad topic only:  accuracy = {accuracy_score(top_test, top_predicted):.3f}")

leaf_predicted = np.empty(len(documents_test), dtype=int)
for topic in np.unique(top_train):
    in_topic = top_train == topic
    leaves = np.unique(train.target[in_topic])
    routed = top_predicted == topic
    if len(leaves) == 1:                             # "for sale" has a single newsgroup
        leaf_predicted[routed] = leaves[0]
        continue
    second = make_pipeline(vectorizer(), ComplementNB(alpha=0.3)).fit(documents_train[in_topic],
                                                                       train.target[in_topic])
    leaf_predicted[routed] = second.predict(documents_test[routed])

print(f"top-down, 20 leaf classes:  accuracy = {accuracy_score(test.target, leaf_predicted):.3f}")
print(f"flat, 20 leaf classes:      accuracy = {accuracy_score(test.target, best):.3f}")

**Read the output:** the dedicated broad-topic classifier (0.825) is no better than reading the broad topic off
the flat model (0.832, Step 6), and the top-down classifier is **worse** than the flat one on the 20 leaves (0.686
against 0.711).

The reason is **error propagation**. A post sent to the wrong broad topic in stage 1 can never be put right in
stage 2: the second-level classifier only knows the leaves of the topic it was given. Every stage-1 mistake is a
guaranteed leaf mistake. Top-down classification pays off when the hierarchy is large (thousands of leaves,
where one flat model is impractical) or when the broad topic is much easier than the leaves. Here, with 20
classes, a single flat model is simpler and better.

## Step 8: What does each class look for?

The SVM's one-vs-rest classifiers each have one weight per word: the words with the largest positive weights are
the strongest evidence *for* that class against all the others.

In [ ]:
words = svm[0].get_feature_names_out()
for group in ["rec.autos", "rec.motorcycles", "sci.space", "talk.politics.guns", "comp.graphics", "comp.windows.x"]:
    weights = svm[-1].coef_[names.tolist().index(group)]
    print(f"{group:20s} {', '.join(words[np.argsort(weights)[::-1][:10]])}")

**Read the output:** each list is recognisably its topic: car, oil, dealer and car makes for `rec.autos`; bike,
ride, helmet and harley for `rec.motorcycles`; orbit, launch, nasa and moon for `sci.space`; firearms, nra and atf
for `talk.politics.guns`. `comp.graphics` (3d, image, tiff) and `comp.windows.x` (motif, widget, xterm, x11) have
distinct top words; their confusions come from posts that use only general computing words both groups share.

Some strong words are not about the topic at all: "dod" (a motorcycle club whose members signed their posts with
their member number), and "jmd" and "bd" in `talk.politics.guns`, which are initials of frequent posters. Even with
headers and signatures removed, traces of *who* wrote a post remain.

## Step 9: Multi-label classification

So far each post has **exactly one** class (*multi-class*). In a **multi-label** problem each item can have **any
number** of labels: a news article tagged both "politics" and "economy", a film that is both "comedy" and
"romance", a support ticket about "billing" and "login". The standard approach is **binary relevance**: one
independent yes/no classifier per label (`sklearn.multiclass.OneVsRestClassifier` with a 0/1 label matrix from
`MultiLabelBinarizer`), each with its own threshold. Metrics change too: the share of items with *all* labels
right is very strict, so per-label F1 or the **Hamming loss** (the share of wrong yes/no decisions) are used.

The newsgroups data is single-label, so this session does not build one. The exercises include a way to try it.

## Step 10: How much do the headers leak?

Finally, measure the leak from Step 2: train and test the same SVM pipeline on the posts **with** headers,
footers and quotes.

In [ ]:
raw_test = fetch_20newsgroups(subset="test")
leaky = make_pipeline(vectorizer(), LinearSVC()).fit(raw.data, raw.target)
print(f"with headers, footers and quotes: accuracy = {accuracy_score(raw_test.target, leaky.predict(raw_test.data)):.3f}")
print(f"content only (Step 3):            accuracy = {accuracy_score(test.target, predictions['LinearSVC (one-vs-rest)']):.3f}")

**Read the output:** with the headers, accuracy jumps from 0.692 to 0.858. Much of that gain comes from
recognising regular posters, organisations and the newsgroup names that appear in cross-posting headers. It is
real information in this archive, but not information the model would have when classifying new text from new
people. Many published results on this dataset use the headers, which is why they look far better than ours.

## What this session hands to the next

- **Multi-class classification** with one-vs-rest and naturally multi-class models; macro F1; per-class reports;
  normalised confusion matrices.
- **Hierarchies:** errors cluster among sibling classes; top-down classifiers propagate errors and need to earn
  their complexity.
- **Leakage in text:** metadata that identifies the class must be removed.
- Session 9 returns to tabular data for the Part 2 bootcamp: comparing many classifiers on a realistic, imbalanced
  business problem.

## Common mistakes and fixes

| Mistake | What goes wrong | Fix |
| --- | --- | --- |
| Keeping email headers, signatures or IDs in the text | The model learns who wrote it, not what it says | Strip metadata; check the top features |
| Reporting accuracy only with many classes | Weak small classes are hidden | Report macro F1 and the per-class report |
| Reading a raw-count confusion matrix | Big classes dominate the colours | `normalize="true"` |
| Building a top-down hierarchy by default | Stage-1 errors cannot be recovered | Compare against a flat model first |
| Treating a multi-label problem as multi-class | Items with two labels are forced into one | One binary classifier per label (`OneVsRestClassifier`) |
| Random splits for posts or messages over time | Future topics leak into training | Split by date, as the "by date" split does |

## Exercises

- [ ] Use `LinearSVC(C=0.3)` and `ComplementNB(alpha=0.1)`. Does either close the gap to the other?
- [ ] Remove `stop_words="english"` and `min_df=2` from the vectoriser. What changes, and why?
- [ ] Build a **multi-label** problem: give every post two labels, its newsgroup's top-level topic and "long" or
  "short" (more or fewer than 100 words). Train `OneVsRestClassifier(LinearSVC())` on a `MultiLabelBinarizer`
  matrix and report the Hamming loss (`sklearn.metrics.hamming_loss`).
- [ ] Train only on the four groups `alt.atheism`, `soc.religion.christian`, `talk.religion.misc`, `sci.space`. How
  well can the model separate the three religion groups?

Next, Session 9 is the Part 2 bootcamp: predicting which bank customers will subscribe to a term deposit, and
comparing seven classifiers fairly.